In [1]:
#!/usr/bin/env python3
"""
Inline-interruption rendering for NoXi transcripts.

Goal
----
Render overlapping turns like:
    <SPK1> Bonjour, tu <SPK2> Oui, je t'entends. <SPK1> m'entends ?,

Heuristic
---------
If utterance B starts before utterance A ends (and speakers differ), and
the overlap (A.end - B.start) >= min_overlap, then:
  - Estimate the cut point within A: ratio r = (B.start - A.start) / (A.end - A.start)
  - Split A's text near that ratio on a word boundary
  - Emit: <SPK_A> A_left , <SPK_B> B_text , <SPK_A> A_right
If A_left or A_right is empty, we skip that fragment.

Inputs
------
Transcripts exported under:
  ~/.Noxi/noxi_exports/transcripts/<SESSION>_expert.audio.transcript.annotation.csv
  ~/.Noxi/noxi_exports/transcripts/<SESSION>_novice.audio.transcript.annotation.csv

Outputs
-------
One file per session:
  ~/.Noxi/noxi_exports/combined_sentence/<SESSION>_sentence_inlineINT.txt

Usage
-----
python combine_with_inline_interruptions.py \
  --transcripts ~/.Noxi/noxi_exports/transcripts \
  --out ~/.Noxi/noxi_exports/combined_sentence \
  --min-overlap-ms 150
"""
import argparse
import csv
import math
import re
from pathlib import Path

def read_transcript_csv(path: Path, tag: str, encoding="utf-8"):
    rows = []
    with path.open("r", encoding=encoding, newline="") as f:
        reader = csv.reader(f, delimiter=";")
        for row in reader:
            if len(row) < 3:
                continue
            try:
                start = float(row[0].strip().replace(",", "."))
                end = float(row[1].strip().replace(",", "."))
            except ValueError:
                continue
            text = row[2].strip()
            if text:
                rows.append([start, end, tag, text])
    return rows

def split_text_at_ratio(text: str, ratio: float):
    """
    Split text into two parts at a position close to `ratio` of the token count.
    We split on whitespace tokens while preserving punctuation in the token.
    """
    if ratio <= 0.0:
        return ("", text)
    # Tokenize coarsely on whitespace
    tokens = text.split()
    if len(tokens) <= 1:
        # Can't split sensibly
        return (text, "")
    # pick an index in [1, len(tokens)-1]
    idx = max(1, min(len(tokens)-1, int(round(ratio * len(tokens)))))
    left = " ".join(tokens[:idx]).strip()
    right = " ".join(tokens[idx:]).strip()
    return (left, right)

def render_inline(sorted_rows, min_overlap=0.150):
    """
    Walk through sorted rows and expand overlaps inline.
    Returns a list of (tag, text) in display order with inline splits.
    """
    out = []
    i = 0
    N = len(sorted_rows)
    while i < N:
        s, e, tag, text = sorted_rows[i]
        # Look ahead to next utterance to check for interruption
        if i + 1 < N:
            ns, ne, ntag, ntext = sorted_rows[i+1]
            if tag != ntag and ns < e < ne and (e - ns) >= min_overlap:
                # Inline interruption: estimate cut point in current text
                dur = max(1e-6, e - s)
                r = max(0.0, min(1.0, (ns - s) / dur))
                left, right = split_text_at_ratio(text, r)
                if left:
                    out.append((tag, left))
                # insert interrupter full text
                out.append((ntag, ntext))
                if right:
                    out.append((tag, right))
                i += 2
                continue
        # default: no interruption here
        out.append((tag, text))
        i += 1
    return out

def to_single_line(tag_text_pairs):
    return " ".join(f"{tag} {txt}" for tag, txt in tag_text_pairs if txt)

In [2]:
encoding="utf-8"
min_overlap_ms=150.0
tdir = Path("./transcripts").expanduser().resolve()
outdir = Path("./combined_sentence").expanduser().resolve()
outdir.mkdir(parents=True, exist_ok=True)

sessions = {}
for p in tdir.glob("*_*.audio.transcript.annotation.csv"):
    session = p.name.split("_", 1)[0]
    if "expert.audio.transcript.annotation.csv" in p.name:
        sessions.setdefault(session, {})["expert"] = p
    elif "novice.audio.transcript.annotation.csv" in p.name:
        sessions.setdefault(session, {})["novice"] = p

processed = 0
min_overlap = min_overlap_ms / 1000.0

for session, files in sorted(sessions.items()):
    if "expert" not in files or "novice" not in files:
        continue
    rows = []
    rows += read_transcript_csv(files["expert"], "<SPK1>", encoding)
    rows += read_transcript_csv(files["novice"], "<SPK2>", encoding)
    rows.sort(key=lambda x: (x[0], x[1]))

    pairs = render_inline(rows, min_overlap=min_overlap)
    sentence = to_single_line(pairs)
    out_path = outdir / f"{session}_sentence.txt"
    out_path.write_text(sentence, encoding=encoding)
    print(f"Wrote {out_path} (segments={len(pairs)})")
    processed += 1

if processed == 0:
    print(f"No sessions with both expert & novice transcripts under {tdir}")

Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/001_sentence.txt (segments=299)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/002_sentence.txt (segments=643)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/003_sentence.txt (segments=172)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/006_sentence.txt (segments=390)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/007_sentence.txt (segments=591)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/009_sentence.txt (segments=249)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/011_sentence.txt (segments=463)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/012_sentence.txt (segments=319)
Wrote /u/sebono/conversational_dominance/data/processed/Noxi/combined_sentence/013_sentence.txt (segments=384)
W

In [3]:
#!/usr/bin/env python3
"""
Collect all combined one-line conversation texts into a single CSV.

Assumes you have per-session files like:
  ~/.Noxi/noxi_exports/combined_sentence/001_sentence.txt
  ~/.Noxi/noxi_exports/combined_sentence/026_sentence.txt
  ...

The CSV will have two columns:
  file_name,text
where file_name is the session folder name (e.g., "001") and text is the
entire concatenated conversation line.

Usage:
  python export_combined_to_csv.py
  # or specify paths:
  python export_combined_to_csv.py --in ~/.Noxi/noxi_exports/combined_sentence \
                                   --out ~/.Noxi/noxi_exports/combined_all.csv
"""
import argparse
import csv
import re
from pathlib import Path

def session_from_filename(name: str) -> str:
    # Accept leading digits as session id (e.g., 001, 026, 150)
    m = re.match(r"^([0-9]{3,})", name)
    return m.group(1) if m else name
indir = Path("./combined_sentence").expanduser().resolve()
outfile = Path("./conversations.csv").expanduser().resolve()
outfile.parent.mkdir(parents=True, exist_ok=True)

rows = []
for p in sorted(indir.glob("*_sentence.txt")):
    session = session_from_filename(p.stem)  # stem like "001_sentence"
    # remove trailing suffix if present
    session = session.replace("_sentence", "")
    text = p.read_text(encoding=encoding).strip()
    rows.append({"file_name": f"S{session}", "file_content": text})

if not rows:
    print(f"No *_sentence.txt files found under {indir}")

with outfile.open("w", encoding=encoding, newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["file_name", "file_content"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {outfile}")

Wrote 76 rows to /u/sebono/conversational_dominance/data/processed/Noxi/conversations.csv


In [4]:
import pandas as pd
dataset = pd.read_csv("/u/sebono/conversational_dominance/data/processed/Noxi/conversations.csv")

In [5]:
#!/usr/bin/env python3
"""
Split a list of conversation IDs into N evenly sized CSV files.

Input:
  - A CSV with a column containing the IDs (default column name: file_name).
    Example row: S001
Output:
  - group_1.csv, ..., group_N.csv with header: conversation_id

Usage:
  python split_into_groups.py --in data/processed/CANDOR/conversations.csv \
                              --column file_name \
                              --groups 8 \
                              --outdir data/processed/CANDOR

Options:
  --shuffle       Randomize order before splitting (off by default).
  --seed 42       Seed used when shuffling.
"""
import argparse
import csv
import math
import random
from pathlib import Path


def split_even(ids, n_groups: int):
    """
    Split ids into n_groups, distributing remainder one-by-one to the first groups.
    """
    total = len(ids)
    base = total // n_groups
    rem = total % n_groups
    groups = []
    start = 0
    for i in range(n_groups):
        size = base + (1 if i < rem else 0)
        groups.append(ids[start:start+size])
        start += size
    return groups

def write_groups(groups, outdir: Path, encoding="utf-8"):
    outdir.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, g in enumerate(groups, start=1):
        p = outdir / f"group_{i}.csv"
        with p.open("w", encoding=encoding, newline="") as f:
            w = csv.writer(f)
            w.writerow(["conversation_id"])
            for x in g:
                w.writerow([x])
        paths.append(p)
    return paths


ids = list(dataset["file_name"])
groups = 4
seed=42
outdir = "./"
shuffle=True
if not ids:
    raise SystemExit("No IDs found in the input file.")

if shuffle:
    random.Random(seed).shuffle(ids)

groups = split_even(ids, groups)
paths = write_groups(groups, Path(outdir), encoding=encoding)
print(f"Wrote {len(paths)} files:")
for p in paths:
    print(" -", p)


Wrote 4 files:
 - group_1.csv
 - group_2.csv
 - group_3.csv
 - group_4.csv


In [6]:
len(ids)

76